# PathSnap: Logistic Regression on the existing preprocessed data

This notebook trains and evaluates **Logistic Regression** using the saved `shared_data/train_candidates.npz` and `shared_data/student_test_candidates.npz` tables.

It does **not** rerun preprocessing. It uses the same six features and candidate labels as the Random Forest and KNN notebooks. Since Logistic Regression is sensitive to feature scales, each fold fits a `StandardScaler` using only that fold's training trips, followed by Logistic Regression.

The notebook performs five-fold cross-validation grouped by trip, fits a final model on all training rows, evaluates the student-test partition, and saves separate Logistic Regression outputs.


In [1]:
from pathlib import Path
import json
import time
import numpy as np
from joblib import dump
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, log_loss
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Find this portable bundle whether Jupyter starts in the bundle or repo root.
start = Path.cwd().resolve()
roots = [start, *start.parents]
bundle_candidates = [start / "PES2UG25CS810_LogisticRegression"]
bundle_candidates.extend(p for p in roots if p.name == "PES2UG25CS810_LogisticRegression")
BUNDLE = next(
    (p for p in bundle_candidates if (p / "03_logistic_regression_same_preprocessed_data.ipynb").is_file()),
    None,
)
if BUNDLE is None:
    raise FileNotFoundError(
        "Could not locate PES2UG25CS810_LogisticRegression. Open this notebook from its folder or the repository root."
    )

data_candidates = [
    BUNDLE / "shared_data",
    BUNDLE.parent / "shared_data",
    BUNDLE.parent / "pathsnap" / "shared_data",
    BUNDLE.parent.parent / "pathsnap" / "shared_data",
]
required_files = [
    "train_candidates.npz", "train_metadata.json",
    "student_test_candidates.npz", "student_test_metadata.json",
]
DATA = next(
    (p for p in data_candidates if all((p / name).is_file() for name in required_files)),
    None,
)
if DATA is None:
    raise FileNotFoundError(
        "Preprocessed files not found. Copy the four train/student-test NPZ and metadata files into this folder's shared_data/. See README.md."
    )
RESULTS = BUNDLE / "results"
ARTIFACTS = BUNDLE / "artifacts"
RESULTS.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)

train_file = np.load(DATA / "train_candidates.npz", allow_pickle=False)
test_file = np.load(DATA / "student_test_candidates.npz", allow_pickle=False)
train_meta = json.loads((DATA / "train_metadata.json").read_text())
test_meta = json.loads((DATA / "student_test_metadata.json").read_text())

X = train_file["X"].astype(np.float32, copy=False)
y = train_file["y"]
groups = train_file["trip_id"]
train_obs = train_file["gps_observation_id"]
X_test = test_file["X"].astype(np.float32, copy=False)
y_test = test_file["y"]
test_obs = test_file["gps_observation_id"]
feature_names = train_file["feature_names"].astype(str).tolist()

assert X.shape == (1_145_304, 6), f"Unexpected train data shape: {X.shape}"
assert X_test.shape == (450_648, 6), f"Unexpected student-test data shape: {X_test.shape}"
assert feature_names == train_meta["feature_names"]

print("Logistic Regression bundle:", BUNDLE)
print("Preprocessed data:", DATA)
print("Train X:", X.shape, "Student-test X:", X_test.shape)
print("Features:", feature_names)
print("Train trips:", len(np.unique(groups)))
print("Train sampled GPS points:", train_meta["sampled_or_total_gps_points"])
print("Student-test GPS points:", test_meta["sampled_or_total_gps_points"])


Project: /media/andy/7CF6C4F2F6C4AE28/Users/Andy/Desktop/COLLEGE/ML/hackathon/pathsnap
Train X: (1145304, 6) Student-test X: (450648, 6)
Features: ['cross_track_m', 'heading_error_rad', 'along_fraction', 'segment_length_m', 'gps_speed_mps', 'time_gap_s']
Train trips: 80
Train sampled GPS points: 49664
Student-test GPS points: 19160


## Why use scaling with Logistic Regression?

Logistic Regression learns a linear decision boundary and estimates a probability for each candidate row. Here it separates positive candidate rows from negative candidate rows using a weighted combination of the six features.

The features use different units and ranges (for example, metres, radians, and seconds). `StandardScaler` centres and scales them so a feature does not dominate just because its numbers are larger. The scaler is inside a pipeline and is refitted within each cross-validation fold; this prevents validation data from leaking into preprocessing.

The positive class is rare: for each covered observation there is one positive among 24 candidates. `class_weight="balanced"` compensates for this class imbalance. Logistic Regression is fast and interpretable, but its decision boundary is linear, so it may not capture complex interactions as well as tree ensembles.


In [2]:
PARAMS = {
    "C": 1.0,
    "class_weight": "balanced",
    "max_iter": 250,
    "solver": "lbfgs",
}
N_SPLITS = 5
BATCH_SIZE = 50_000


def make_model():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(**PARAMS),
    )


def predict_probabilities_batched(model, features, batch_size=BATCH_SIZE):
    predictions = []
    for start in range(0, len(features), batch_size):
        stop = min(start + batch_size, len(features))
        predictions.append(model.predict_proba(features[start:stop])[:, 1])
    return np.concatenate(predictions)


def observation_top1(y_true, probabilities, observation_ids):
    """Choose the highest-probability candidate per GPS point; stable tie handling."""
    _, inverse = np.unique(observation_ids, return_inverse=True)
    n_observations = int(inverse.max()) + 1
    best_probability = np.full(n_observations, -np.inf)
    np.maximum.at(best_probability, inverse, probabilities)
    is_winner = probabilities == best_probability[inverse]
    winning_rows = np.flatnonzero(is_winner)
    first_winner = np.full(n_observations, len(y_true), dtype=np.int64)
    np.minimum.at(first_winner, inverse[winning_rows], winning_rows)
    return float(np.mean(y_true[first_winner] == 1)), n_observations


def candidate_macro_f1(y_true, probabilities):
    return float(f1_score(y_true, probabilities >= 0.5, average="macro"))

print("Logistic Regression parameters:", PARAMS)
print("CV folds:", N_SPLITS)


Logistic Regression parameters: {'C': 1.0, 'class_weight': 'balanced', 'max_iter': 250, 'solver': 'lbfgs'}
CV folds: 5


## 1. Five-fold cross-validation by trip

`GroupKFold` keeps all rows from a trip together. For each fold, the pipeline learns scaling values and classifier parameters using the training trips, then scores the held-out trips. This avoids putting near-duplicate observations from one trip on both sides of a split.

This cell saves fold-level measurements to `results/logistic_regression_cv.json`.


In [3]:
cv = GroupKFold(n_splits=N_SPLITS)
fold_results = []
cv_started = time.perf_counter()

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
    print(f"Fold {fold}/{N_SPLITS}: fit rows={len(train_idx):,}; validation rows={len(valid_idx):,}", flush=True)
    model = make_model()
    fit_started = time.perf_counter()
    model.fit(X[train_idx], y[train_idx])
    fit_seconds = time.perf_counter() - fit_started

    predict_started = time.perf_counter()
    valid_probabilities = predict_probabilities_batched(model, X[valid_idx])
    predict_seconds = time.perf_counter() - predict_started
    top1, n_observations = observation_top1(y[valid_idx], valid_probabilities, train_obs[valid_idx])
    macro_f1 = candidate_macro_f1(y[valid_idx], valid_probabilities)
    result = {
        "fold": fold,
        "held_out_trips": sorted(np.unique(groups[valid_idx]).astype(str).tolist()),
        "candidate_rows": int(len(valid_idx)),
        "observations": int(n_observations),
        "observation_top1": top1,
        "candidate_macro_f1_at_0.5": macro_f1,
        "fit_seconds": fit_seconds,
        "prediction_seconds": predict_seconds,
    }
    fold_results.append(result)
    print(result, flush=True)

cv_seconds = time.perf_counter() - cv_started
cv_summary = {
    "model": "LogisticRegression",
    "parameters": PARAMS,
    "features": feature_names,
    "train_rows": int(len(y)),
    "train_trips": int(len(np.unique(groups))),
    "cv_method": "5-fold GroupKFold by trip_id",
    "observation_top1_mean": float(np.mean([r["observation_top1"] for r in fold_results])),
    "observation_top1_std": float(np.std([r["observation_top1"] for r in fold_results])),
    "candidate_macro_f1_mean": float(np.mean([r["candidate_macro_f1_at_0.5"] for r in fold_results])),
    "folds": fold_results,
    "total_cv_seconds": cv_seconds,
}
(RESULTS / "logistic_regression_cv.json").write_text(json.dumps(cv_summary, indent=2))
print("CV summary:")
print(json.dumps({k: v for k, v in cv_summary.items() if k != "folds"}, indent=2))


Fold 1/5: fit rows=917,400; validation rows=227,904
{'fold': 1, 'held_out_trips': ['00000006', '00000012', '00000019', '00000021', '00000031', '00000033', '00000042', '00000044', '00000053', '00000063', '00000074', '00000079', '00000080', '00000082', '00000083', '00000084'], 'candidate_rows': 227904, 'observations': 9496, 'observation_top1': 0.7114574557708508, 'candidate_macro_f1_at_0.5': 0.6753730800055324, 'fit_seconds': 2.534451233001164, 'prediction_seconds': 0.04732992799836211}
Fold 2/5: fit rows=915,144; validation rows=230,160
{'fold': 2, 'held_out_trips': ['00000005', '00000008', '00000011', '00000013', '00000014', '00000022', '00000029', '00000043', '00000058', '00000060', '00000062', '00000069', '00000075', '00000092', '00000097', '00000099'], 'candidate_rows': 230160, 'observations': 9590, 'observation_top1': 0.6856100104275287, 'candidate_macro_f1_at_0.5': 0.6384952947317772, 'fit_seconds': 2.795362534998276, 'prediction_seconds': 0.019673514001624426}
Fold 3/5: fit rows=

## 2. Fit on all training rows and evaluate student test

After cross-validation, fit a fresh scaler + Logistic Regression pipeline on all training candidate rows. Then score every available student-test candidate. The student-test partition is for development evaluation; the reserved evaluator partition must remain untouched.

This cell saves `artifacts/logistic_regression.joblib` and `results/logistic_regression_results.json`.


In [4]:
final_model = make_model()
fit_started = time.perf_counter()
final_model.fit(X, y)
final_fit_seconds = time.perf_counter() - fit_started
dump(final_model, ARTIFACTS / "logistic_regression.joblib")
print(f"Saved model: {ARTIFACTS / 'logistic_regression.joblib'}")

infer_started = time.perf_counter()
test_probabilities = predict_probabilities_batched(final_model, X_test)
inference_seconds = time.perf_counter() - infer_started
student_top1, covered_observations = observation_top1(y_test, test_probabilities, test_obs)
student_macro_f1 = candidate_macro_f1(y_test, test_probabilities)
student_log_loss = float(log_loss(y_test, np.c_[1 - test_probabilities, test_probabilities], labels=[0, 1]))

results = {
    "model": "LogisticRegression",
    "parameters": PARAMS,
    "features": feature_names,
    "train_candidate_rows": int(len(y)),
    "train_gps_points": int(train_meta["sampled_or_total_gps_points"]),
    "train_candidate_miss_rate": float(train_meta["candidate_miss_rate"]),
    "student_test_candidate_rows": int(len(y_test)),
    "student_test_gps_points": int(test_meta["sampled_or_total_gps_points"]),
    "student_test_covered_points": int(covered_observations),
    "student_test_candidate_miss_rate": float(test_meta["candidate_miss_rate"]),
    "grouped_5fold_cv_observation_top1_mean": cv_summary["observation_top1_mean"],
    "grouped_5fold_cv_observation_top1_std": cv_summary["observation_top1_std"],
    "grouped_5fold_cv_candidate_macro_f1_mean": cv_summary["candidate_macro_f1_mean"],
    "student_test_observation_top1": student_top1,
    "student_test_candidate_macro_f1_at_0.5": student_macro_f1,
    "student_test_candidate_log_loss": student_log_loss,
    "final_fit_seconds": final_fit_seconds,
    "student_test_inference_seconds": inference_seconds,
    "artifact": "artifacts/logistic_regression.joblib",
    "evaluation_note": "Top-1 is conditional on a positive reference candidate being generated. Labels are pointwise geometric pseudo-labels; these are candidate-ranking results, not sequence-level route accuracy.",
}
(RESULTS / "logistic_regression_results.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))


Saved model: /media/andy/7CF6C4F2F6C4AE28/Users/Andy/Desktop/COLLEGE/ML/hackathon/pathsnap/artifacts/logistic_regression.joblib
{
  "model": "LogisticRegression",
  "parameters": {
    "C": 1.0,
    "class_weight": "balanced",
    "max_iter": 250,
    "solver": "lbfgs"
  },
  "features": [
    "cross_track_m",
    "heading_error_rad",
    "along_fraction",
    "segment_length_m",
    "gps_speed_mps",
    "time_gap_s"
  ],
  "train_candidate_rows": 1145304,
  "train_gps_points": 49664,
  "train_candidate_miss_rate": 0.03912290592783505,
  "student_test_candidate_rows": 450648,
  "student_test_gps_points": 19160,
  "student_test_covered_points": 18777,
  "student_test_candidate_miss_rate": 0.01998956158663883,
  "grouped_5fold_cv_observation_top1_mean": 0.707689654522902,
  "grouped_5fold_cv_observation_top1_std": 0.02841920059265555,
  "grouped_5fold_cv_candidate_macro_f1_mean": 0.6572791826787406,
  "student_test_observation_top1": 0.7095382648985461,
  "student_test_candidate_macro_f1

## 3. Interpreting the metrics

- **Observation top-1** measures whether the model ranks the positive road candidate first for a covered GPS point.
- **Candidate macro-F1** evaluates positive/negative candidate classification at a 0.5 threshold, weighting both classes equally.
- **Candidate miss rate** is reported separately because the classifier cannot choose a reference road that candidate generation did not include.
- **Log loss** checks the quality of predicted probabilities; lower is better.

These results describe pointwise road-candidate scoring with geometric pseudo-labels. They do not measure whether the sequence of selected road arcs connects into a valid route. HMM/Viterbi decoding is a later step.

Run the notebook with the **Python (PathSnap local)** kernel and choose **Run All**. It loads the existing processed files, performs five-fold validation, trains the final model, and writes the artifact/JSON outputs described above.
